# 📚 RAG Pipeline: Lisansüstü Yönetmeliği Üzerinde Soru-Cevap

Bu notebook, `lisansustu.pdf` dosyasından ChromaDB vektör veritabanı oluşturur ve  
Anthropic Claude API (claude-haiku-3-5) kullanarak RAG tabanlı soru-cevap sistemi kurar.

**Pipeline Adımları:**
1. PDF → Metin çıkarma (pdfplumber)
2. Metin → Chunk'lara bölme (karakter + token bazlı hibrit)
3. Chunk'lar → Vektörleştirme + ChromaDB'ye kaydetme
4. Sorgu → Anlamsal erişim (semantic retrieval)
5. Bağlam + Sorgu → Claude API ile yanıt üretme

## 🔧 1. Gerekli Kütüphanelerin Kurulumu

In [ ]:
# Gerekli tüm kütüphaneler kuruluyor
%pip install -q pdfplumber langchain langchain-text-splitters sentence-transformers chromadb anthropic opentelemetry-sdk opentelemetry-api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6

## 📦 2. Import ve Sabitler

In [ ]:
# Tüm kütüphaneler import ediliyor; PDF yolu ve model adları sabit olarak tanımlanıyor
import os
import pdfplumber
import chromadb
import anthropic
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter, SentenceTransformersTokenTextSplitter

# --- SABİTLER ---
PDF_PATH          = "lisansustu.pdf"          # PDF dosyasının yolu (notebook ile aynı dizinde)
PDF_TITLE         = "lisansustu.pdf"
PDF_CATEGORY      = "Egitim Yonergeleri"

EMBEDDING_MODEL   = "distiluse-base-multilingual-cased-v1"  # Çok dilli embedding modeli
CLAUDE_MODEL      = "claude-haiku-4-5"                       # Yanıt üretecek LLM

CHUNK_SIZE_CHAR   = 1500   # Karakter bazlı bölme boyutu
CHUNK_OVERLAP_CHAR= 200    # Karakter bazlı örtüşme
TOKENS_PER_CHUNK  = 128    # Token bazlı maksimum token sayısı
TOKEN_OVERLAP     = 10     # Token bazlı örtüşme

COLLECTION_NAME   = "lisansustu_collection"
VECTOR_DB_PATH    = "./chromadb"
TOP_K             = 3      # Sorguda döndürülecek en yakın chunk sayısı

print("✅ Sabitler tanımlandı")
print(f"   PDF     : {PDF_PATH}")
print(f"   Embedding: {EMBEDDING_MODEL}")
print(f"   LLM     : {CLAUDE_MODEL}")

✅ Sabitler tanımlandı
   PDF     : lisansustu.pdf
   Embedding: distiluse-base-multilingual-cased-v1
   LLM     : claude-haiku-4-5


## 📄 3. PDF'ten Metin Çıkarma

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Dosyanın tam yolu
PDF_PATH = "/content/drive/MyDrive/BTK/lisansustu.pdf"

# Kontrol et
if os.path.exists(PDF_PATH):
    print(f"✅ Dosya bulundu: {PDF_PATH}")
    print(f"📦 Boyut: {os.path.getsize(PDF_PATH):,} bytes")
else:
    print(f"❌ Dosya bulunamadı: {PDF_PATH}")

✅ Dosya bulundu: /content/drive/MyDrive/BTK/lisansustu.pdf
📦 Boyut: 472,129 bytes


In [ ]:
# pdfplumber ile PDF okunuyor; header/footer satırları temizleniyor

HEADER_KEYWORDS = [
    'Senato Karar No', 'Revizyon Tarihi', 'Revizyon No',
    'İlk Yayın Tarihi', 'Sayfa:'
]

def clean_page(text: str) -> str:
    """Sayfa metninden header/footer satırlarını temizler."""
    lines = text.split('\n')
    cleaned = [line for line in lines
               if not any(kw in line for kw in HEADER_KEYWORDS)]
    return '\n'.join(cleaned)

def load_pdf(PDF_PATH: str) -> list[str]:
    """PDF'teki her sayfayı temizlenmiş metin olarak döndürür."""
    if not os.path.exists(PDF_PATH):
        raise FileNotFoundError(f"❌ PDF bulunamadı: {PDF_PATH}")

    pages = []
    with pdfplumber.open(PDF_PATH) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text and text.strip():
                pages.append(clean_page(text.strip()))

    print(f"📗 '{os.path.basename(PDF_PATH)}' yüklendi → {len(pages)} sayfa")
    return pages

pdf_pages = load_pdf(PDF_PATH)

📗 'lisansustu.pdf' yüklendi → 18 sayfa


## ✂️ 4. Hibrit Chunking: Karakter → Token

In [ ]:
# Adım 1: Karakter bazlı bölme (büyük metin → orta boy parçalar)
# Adım 2: Token bazlı bölme (orta boy → model sınırına uygun küçük parçalar)
# İki aşamalı hibrit yaklaşım hem hızlı hem de embedding modeli için güvenlidir.

def split_by_characters(pages: list[str],
                         chunk_size: int = CHUNK_SIZE_CHAR,
                         chunk_overlap: int = CHUNK_OVERLAP_CHAR) -> list[str]:
    """Sayfaları karakter bazlı parçalara böler."""
    splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " ", ""],
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_text('\n\n'.join(pages))
    print(f"   Karakter bazlı bölme → {len(chunks)} parça (chunk_size={chunk_size})")
    return chunks

def split_by_tokens(char_chunks: list[str],
                    model_name: str = EMBEDDING_MODEL,
                    tokens_per_chunk: int = TOKENS_PER_CHUNK,
                    chunk_overlap: int = TOKEN_OVERLAP) -> list[str]:
    """Karakter parçalarını token bazlı olarak yeniden böler."""
    splitter = SentenceTransformersTokenTextSplitter(
        model_name=model_name,
        tokens_per_chunk=tokens_per_chunk,
        chunk_overlap=chunk_overlap
    )
    token_chunks = []
    for chunk in char_chunks:
        token_chunks.extend(splitter.split_text(chunk))
    print(f"   Token bazlı bölme   → {len(token_chunks)} parça (tokens_per_chunk={tokens_per_chunk})")
    return token_chunks

print("✂️ Chunking başlıyor...")
char_chunks  = split_by_characters(pdf_pages)
token_chunks = split_by_tokens(char_chunks)
print(f"\n📊 Özet: {len(pdf_pages)} sayfa → {len(char_chunks)} karakter chunk → {len(token_chunks)} token chunk")

✂️ Chunking başlıyor...
   Karakter bazlı bölme → 59 parça (chunk_size=1500)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

   Token bazlı bölme   → 216 parça (tokens_per_chunk=128)

📊 Özet: 18 sayfa → 59 karakter chunk → 216 token chunk


## 🗄️ 5. ChromaDB Vektör Veritabanı Oluşturma

In [ ]:
# Embedding fonksiyonu ve ChromaDB koleksiyonu oluşturuluyor.
# Embedding fonksiyonu koleksiyona bağlandığı için add() ve query()
# çağrılarında metinler otomatik olarak vektörleştirilir.

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL
)

def get_or_create_collection(db_path: str,
                              collection_name: str,
                              emb_fn,
                              reset: bool = False):
    """ChromaDB istemcisi ve koleksiyonu oluşturur/getirir."""
    client = chromadb.PersistentClient(path=db_path)

    existing = [c.name for c in client.list_collections()]
    if collection_name in existing:
        if reset:
            client.delete_collection(collection_name)
            print(f"🗑️  '{collection_name}' koleksiyonu silindi ve yeniden oluşturuluyor...")
            collection = client.create_collection(collection_name, embedding_function=emb_fn)
        else:
            collection = client.get_collection(collection_name, embedding_function=emb_fn)
            print(f"♻️  '{collection_name}' mevcut koleksiyon kullanılıyor ({collection.count()} belge)")
            return client, collection
    else:
        collection = client.create_collection(collection_name, embedding_function=emb_fn)
        print(f"✨ '{collection_name}' yeni koleksiyon oluşturuldu")

    return client, collection

# reset=True → mevcut koleksiyonu silip yeniden oluşturur
# reset=False → zaten doluysa tekrar ekleme yapmaz
chroma_client, chroma_collection = get_or_create_collection(
    VECTOR_DB_PATH, COLLECTION_NAME, embedding_fn, reset=True
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

✨ 'lisansustu_collection' yeni koleksiyon oluşturuldu


## 💾 6. Chunk'ları Veritabanına Ekleme

In [ ]:
# Her chunk için benzersiz bir ID ve kaynak metadata hazırlanıp
# ChromaDB'ye ekleniyor. Embedding işlemi otomatik gerçekleşir.

def add_chunks_to_collection(chunks: list[str],
                              collection,
                              title: str = PDF_TITLE,
                              category: str = PDF_CATEGORY,
                              batch_size: int = 100):
    """Chunk listesini metadata ile birlikte ChromaDB koleksiyonuna ekler."""
    ids       = [str(i) for i in range(len(chunks))]
    metadatas = [{'document': title, 'category': category, 'chunk_index': i}
                 for i in range(len(chunks))]

    # Büyük koleksiyonlarda bellek sorununu önlemek için batch'ler halinde ekle
    for start in range(0, len(chunks), batch_size):
        end = min(start + batch_size, len(chunks))
        collection.add(
            ids=ids[start:end],
            documents=chunks[start:end],
            metadatas=metadatas[start:end]
        )
        print(f"   [{start}–{end}] eklendi", end="\r")

    print(f"\n✅ Toplam {collection.count()} chunk veritabanına eklendi")
    return collection

chroma_collection = add_chunks_to_collection(token_chunks, chroma_collection)

   [200–216] eklendi
✅ Toplam 216 chunk veritabanına eklendi


## 🔍 7. Anlamsal Erişim (Semantic Retrieval)

In [ ]:
# Kullanıcı sorgusu vektörleştirilerek ChromaDB'deki en yakın
# TOP_K chunk getirilir. Bu chunk'lar LLM'e bağlam olarak verilecek.

def retrieve(query: str, collection, top_k: int = TOP_K) -> dict:
    """Sorguya anlamca en yakın chunk'ları döndürür."""
    results = collection.query(
        query_texts=query,
        n_results=top_k
    )
    return results

def format_retrieved_chunks(results: dict) -> str:
    """Getirilen chunk'ları LLM prompt'una uygun tek bir metin bloğuna dönüştürür."""
    docs      = results['documents'][0]
    distances = results['distances'][0]
    lines = []
    for i, (doc, dist) in enumerate(zip(docs, distances), 1):
        lines.append(f"[Bağlam {i} | uzaklık: {dist:.4f}]\n{doc}")
    return "\n\n".join(lines)

# --- Test: Anlamsal erişim ---
test_query = "Seminer sunmak isteyen bir öğrenci ne yapmalıdır?"
results    = retrieve(test_query, chroma_collection)

print(f"🔍 Sorgu: {test_query}\n")
print("📋 Getirilen Bağlamlar:")
print("-" * 60)
for i, (doc, dist) in enumerate(zip(results['documents'][0], results['distances'][0]), 1):
    print(f"\n[{i}] Uzaklık: {dist:.4f}")
    print(doc[:200], "..." if len(doc) > 200 else "")

🔍 Sorgu: Seminer sunmak isteyen bir öğrenci ne yapmalıdır?

📋 Getirilen Bağlamlar:
------------------------------------------------------------

[1] Uzaklık: 0.6498
programı dersleri hariç ) dikkate alınarak hesaplanır. Öğrenci, ders döneminde olmak koşuluyla, GNO [UNK] sunu yükseltmek amacıyla başarılı olduğu dersleri tekrarlayabilir. Tekrar edilen derslerde alı ...

[2] Uzaklık: 0.6780
geçerlidir. Seminer dersinin yürütülmesi ve değerlendirilmesi MADDE 22 - ( 1 ) Seminer dersi, danışman sorumluluğunda yürütülür. Seminer konusu danışman ve öğrencinin ortak kararıyla belirlenir. ( 2 ) ...

[3] Uzaklık: 0.7168
##ınavına girmek zorundadır. ( 4 ) Azami süresi içerisinde tezini teslim etmiş, tez savunmasına girmiş ancak tezinde düzeltme verilmiş öğrencilerin tez düzeltme için verilen sürenin azami süresiyi aşm ...


## 🤖 8. Claude API ile Yanıt Üretme

In [ ]:
# Getirilen bağlam chunk'ları ve kullanıcı sorusu birleştirilerek
# Claude'a prompt olarak gönderilir; model bağlama dayalı yanıt üretir.

# API anahtarı ortam değişkeninden alınıyor.
# Colab'da: os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
os.environ['ANTHROPIC_API_KEY'] = "buraya_api_key_yaz"  # veya boş bırakın

ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
if not ANTHROPIC_API_KEY:
    raise ValueError("❌ ANTHROPIC_API_KEY ortam değişkeni ayarlanmamış!")

claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print(f"✅ Anthropic istemcisi oluşturuldu → model: {CLAUDE_MODEL}")


SYSTEM_PROMPT = """\
Sen Fırat Üniversitesi lisansüstü eğitim yönetmeliği konusunda uzmanlaşmış
bir asistansın. Sana verilen BAĞLAM bölümündeki bilgilere dayanarak
soruları Türkçe olarak yanıtla.

Kurallar:
- Yalnızca bağlamda yer alan bilgileri kullan.
- Bağlamda cevap yoksa bunu açıkça belirt: "Bu bilgi yönetmelikte bulunamadı."
- Yanıtlarını kısa, net ve anlaşılır tut.
- Gerektiğinde madde madde listele.
"""

def generate_answer(query: str,
                    context: str,
                    client=claude_client,
                    model: str = CLAUDE_MODEL,
                    max_tokens: int = 1024) -> str:
    """Bağlam ve soruyu birleştirip Claude'dan yanıt üretir."""
    user_message = f"""BAĞLAM:
{context}

SORU: {query}

Yukarıdaki bağlama dayanarak soruyu yanıtla."""

    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text


# --- Test: Tam RAG pipeline ---
context = format_retrieved_chunks(results)
answer  = generate_answer(test_query, context)

print(f"❓ Soru   : {test_query}")
print("-" * 60)
print(f"💬 Yanıt  :\n{answer}")

✅ Anthropic istemcisi oluşturuldu → model: claude-haiku-4-5
❓ Soru   : Seminer sunmak isteyen bir öğrenci ne yapmalıdır?
------------------------------------------------------------
💬 Yanıt  :
# Seminer Sunmak İsteyen Öğrencinin Yapması Gerekenler

Bağlama dayanarak, seminer sunmak isteyen bir öğrencinin aşağıdaki adımları takip etmesi gerekir:

1. **Danışmanla İş Birliği**: Seminer konusunu danışmanla ortaklaşa belirlemek
   - Seminer dersi danışman sorumluluğunda yürütülür
   - Konusu danışman ve öğrencinin ortak kararıyla belirlenir

2. **Resmî İlan Bekleme**: Seminer sunma tarihi ve yerinin duyurulmasını beklemek
   - Danışmanın önerisi ile bir hafta önceden EABD Başkanı tarafından ilan edilir
   - İlan, akademik takviminde belirtilen tarihler dikkate alınarak yapılır

3. **Belirlenen Tarih ve Yerde Sunmak**: Ilan edilen yer, zaman ve tarihte semineri sunmak


## 🔄 9. Tam RAG Fonksiyonu

In [ ]:
# Retrieval + Generation adımlarını tek çağrıda birleştiren
# rag_ask() fonksiyonu tanımlanıyor.

def rag_ask(query: str,
            collection=chroma_collection,
            top_k: int = TOP_K,
            verbose: bool = True) -> str:
    """
    Tam RAG pipeline:
      1. Sorguya en yakın chunk'ları getir (retrieve)
      2. Bağlamı formatla
      3. Claude'dan yanıt üret (generate)
    """
    # 1. Retrieve
    results = retrieve(query, collection, top_k)
    context = format_retrieved_chunks(results)

    if verbose:
        print(f"🔍 Getirilen chunk sayısı: {len(results['documents'][0])}")
        for i, dist in enumerate(results['distances'][0], 1):
            print(f"   Chunk {i} → uzaklık: {dist:.4f}")
        print()

    # 2. Generate
    answer = generate_answer(query, context)
    return answer

print("✅ rag_ask() fonksiyonu hazır")

✅ rag_ask() fonksiyonu hazır


## 💬 10. Örnek Sorgular

Farklı türde sorular denenerek RAG sisteminin performansı gözlemleniyor.

In [ ]:
# Soru 1: Doğrudan cevabı olan soru
soru = "Doktora yeterlik sınavı ne zaman yapılır?"
print(f"❓ {soru}")
print("=" * 60)
yanit = rag_ask(soru)
print(f"💬 {yanit}")

❓ Doktora yeterlik sınavı ne zaman yapılır?
🔍 Getirilen chunk sayısı: 3
   Chunk 1 → uzaklık: 0.5613
   Chunk 2 → uzaklık: 0.6197
   Chunk 3 → uzaklık: 0.6297

💬 # Doktora Yeterlik Sınavı Zamanlaması

Bağlamda yer alan bilgilere göre:

**Lisans derecesi ile kabul edilmiş olan öğrenci, en geç yedinci yarıyılın sonuna kadar yeterlik sınavına girmek zorundadır.**

Bu süre içerisinde yeterlik sınavına girmeyen öğrencilerin enstitü ile ilişiği kesilir.

**Not:** Bağlamda, doktora yeterlik sınavının tam olarak hangi yarıyılda veya ne zaman başlanması gerektiği hakkında daha detaylı bilgi bulunmamaktadır. Sadece son tarih (7. yarıyılın sonu) belirtilmiştir.


In [ ]:
# Soru 2: Süreç/prosedür sorusu
soru = "Ders grupları ve kontenjanlar kim tarafından belirlenir?"
print(f"❓ {soru}")
print("=" * 60)
yanit = rag_ask(soru)
print(f"💬 {yanit}")

❓ Ders grupları ve kontenjanlar kim tarafından belirlenir?
🔍 Getirilen chunk sayısı: 3
   Chunk 1 → uzaklık: 0.6458
   Chunk 2 → uzaklık: 0.6713
   Chunk 3 → uzaklık: 0.7471

💬 Bu bilgi yönetmelikte bulunamadı.

Verilen bağlam parçaları, jüri oluşumu, başvuru esasları ve diğer konuları içermekle birlikte, ders grupları ve kontenjanlarının kim tarafından belirlendiğine ilişkin bilgi bulunmamaktadır.


In [ ]:
# Soru 3: Bağlamda olmayan bilgi — model bunu belirtmeli
soru = "Öğrencinin muafiyet hakkı kaç kere kullanılabilir?"
print(f"❓ {soru}")
print("=" * 60)
yanit = rag_ask(soru)
print(f"💬 {yanit}")

❓ Öğrencinin muafiyet hakkı kaç kere kullanılabilir?
🔍 Getirilen chunk sayısı: 3
   Chunk 1 → uzaklık: 0.6084
   Chunk 2 → uzaklık: 0.6258
   Chunk 3 → uzaklık: 0.6468

💬 Bu bilgi yönetmelikte bulunamadı.

Verilen bağlamda "muafiyet hakkı" ile ilgili herhangi bir bilgi yer almamaktadır. Bağlamda yalnızca "yatay geçiş hakkı" konusunda bilgi bulunmakta olup, buna göre yatay geçiş hakkı kurum içinde en çok bir defa kullanılabilir.


In [ ]:
# Soru 4: Genel bilgi sorusu
soru = "Tez savunma sınavı için hangi şartlar gerekmektedir?"
print(f"❓ {soru}")
print("=" * 60)
yanit = rag_ask(soru)
print(f"💬 {yanit}")

❓ Tez savunma sınavı için hangi şartlar gerekmektedir?
🔍 Getirilen chunk sayısı: 3
   Chunk 1 → uzaklık: 0.6114
   Chunk 2 → uzaklık: 0.6164
   Chunk 3 → uzaklık: 0.6708

💬 # Tez Savunma Sınavı Şartları

Bağlama dayanarak, tez savunma sınavı için aşağıdaki şartlar gerekmektedir:

## Zaman Şartı
- Jüri üyeleri, tezi aldıktan sonra **en geç bir ay içinde** toplanarak sınavı gerçekleştirmelidir.

## Sınav İçeriği
Tez savunma sınavı iki bölümden oluşur:
1. **Tez çalışmasının sunumu**
2. **Soru-cevap bölümü**

## Katılım Şartı
Sınav **açık ortamda** yapılır ve aşağıdakiler katılabilir:
- Öğretim elemanları
- Lisansüstü öğrenciler
- Alanında uzman dinleyiciler

## Ek Şart
- **EABD Başkanı/Program Koordinatörü**, sınavların dinleyicilere açık şekilde yapılması için gerekli tedbirleri almalıdır.


In [ ]:
# Soru 5: Kendi sorunuzu girin
soru = input("❓ Sorunuzu girin: ")
if soru.strip():
    print("=" * 60)
    yanit = rag_ask(soru)
    print(f"💬 {yanit}")
else:
    print("Soru girilmedi.")

KeyboardInterrupt: Interrupted by user

## 📊 11. Koleksiyon İstatistikleri


In [ ]:
# Veritabanı hakkında özet bilgi yazdırılıyor
print("📊 Koleksiyon Bilgileri")
print("=" * 40)
print(f"Koleksiyon adı : {COLLECTION_NAME}")
print(f"Toplam chunk   : {chroma_collection.count()}")
print(f"Embedding modeli: {EMBEDDING_MODEL}")
print(f"LLM modeli     : {CLAUDE_MODEL}")
print(f"Top-K          : {TOP_K}")
print()

# İlk 3 chunk'ı önizle
sample = chroma_collection.get(['0','1','2'])
print("📄 İlk 3 Chunk Önizleme:")
print("-" * 40)
for i, doc in enumerate(sample['documents'], 1):
    print(f"[{i}] {doc[:150]}...\n")

📊 Koleksiyon Bilgileri
Koleksiyon adı : lisansustu_collection
Toplam chunk   : 216
Embedding modeli: distiluse-base-multilingual-cased-v1
LLM modeli     : claude-haiku-4-5
Top-K          : 3

📄 İlk 3 Chunk Önizleme:
----------------------------------------
[1] 2023 - 2024 / 21. 6 10. 07. 2024 20. 02. 2025 2024 - 2025 / 10. 18 1 / 18 FIRAT ÜNİVERSİTESİ LİSANSÜSTÜ EĞİTİM - ÖĞRETİM YÖNERGESİ BİRİNCİ BÖLÜM Başla...

[2] ##tim ve Öğretim Yönetmeliği ile Fırat Üniversitesi Lisansüstü Eğitim Öğretim ve Sınav Yönetmeliği çerçevesinde Fırat Üniversitesi enstitüleri tarafın...

[3] . Dayanak MADDE 2 - ( 1 ) Bu Yönerge, 2547 sayılı Yükseköğretim Kanunu [UNK] nun 14 / b ve 44 üncü maddeleri, Yükseköğretim Kurulu Başkanlığı tarafınd...



In [ ]:
# Gradio kütüphanesi kuruluyor
%pip install -q gradio

In [ ]:
# Gradio arayüzü: kullanıcı soru yazar, yanıt + kaynak kanıt gösterilir.
# rag_ask_with_source() → yanıt metnini ve en yakın chunk bilgisini ayrı döndürür.

import gradio as gr

def rag_ask_with_source(query: str,
                         collection=chroma_collection,
                         top_k: int = TOP_K) -> tuple[str, str]:
    """
    RAG pipeline + kaynak bilgisi.
    Döndürür: (yanıt_metni, kaynak_notu)
    """
    if not query.strip():
        return "Lütfen bir soru girin.", ""

    # 1. Retrieve
    results  = retrieve(query, collection, top_k)
    context  = format_retrieved_chunks(results)

    # 2. Generate
    answer   = generate_answer(query, context)

    # 3. Kaynak notu: en yakın (en düşük uzaklıklı) chunk
    best_doc  = results['documents'][0][0]
    best_dist = results['distances'][0][0]
    best_meta = results['metadatas'][0][0]

    source_note = (
        f"📎 **Kaynak:** {best_meta.get('document', '—')} "
        f"(chunk #{best_meta.get('chunk_index', '?')}, "
        f"benzerlik uzaklığı: {best_dist:.4f})\n\n"
        f"**En ilgili metin parçası:**\n> {best_doc[:300]}"
        + ("…" if len(best_doc) > 300 else "")
    )

    return answer, source_note


# --- Gradio UI tanımı ---
with gr.Blocks(theme=gr.themes.Soft(), title="Lisansüstü Yönetmelik Asistanı") as demo:

    gr.Markdown(
        """
        # 📚 Lisansüstü Yönetmelik Asistanı
        Yönetmelik hakkında Türkçe sorularınızı sorun.
        Yanıtlar yalnızca belgedeki bilgilere dayanır.
        """
    )

    with gr.Row():
        with gr.Column(scale=3):
            query_box = gr.Textbox(
                label="Sorunuz",
                placeholder="Örnek: Tez savunma sınavı için hangi şartlar gerekmektedir?",
                lines=3
            )
            with gr.Row():
                submit_btn = gr.Button("Yanıtla", variant="primary")
                clear_btn  = gr.ClearButton([query_box], value="Temizle")

        with gr.Column(scale=1):
            top_k_slider = gr.Slider(
                minimum=1, maximum=5, value=TOP_K, step=1,
                label="Kaynak chunk sayısı (Top-K)"
            )

    answer_box = gr.Textbox(
        label="Yanıt",
        lines=6,
        interactive=False
    )

    source_box = gr.Markdown(label="📎 Kaynak Kanıt")

    # Örnek sorular
    gr.Examples(
        examples=[
            ["Muafiyetten feragat eden öğrenci tekrar başvuru yapabilir mi?"],
            ["Tez savunma sınavı için hangi şartlar gerekmektedir?"],
            ["Ders grupları ve kontenjanlar kim tarafından belirlenir?"],
            ["Danışman atama süreci nasıl işler?"],
            ["Öğrencinin muafiyet hakkı kaç kere kullanılabilir?"],
        ],
        inputs=query_box
    )

    # Bağlantılar
    def run(query, top_k):
        return rag_ask_with_source(query, chroma_collection, int(top_k))

    submit_btn.click(
        fn=run,
        inputs=[query_box, top_k_slider],
        outputs=[answer_box, source_box]
    )
    query_box.submit(
        fn=run,
        inputs=[query_box, top_k_slider],
        outputs=[answer_box, source_box]
    )
    clear_btn.click(lambda: ("", ""), outputs=[answer_box, source_box])

# share=True → Colab'da dışarıdan erişilebilir geçici URL oluşturur
demo.launch(share=True, debug=False)

/tmp/ipykernel_612/726000473.py:40: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Lisansüstü Yönetmelik Asistanı") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c7ddf97588ecd952e9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
